# TikZ fine-tuning

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

dataset_name = "lzhoang2601/mathematical-tikz"
dataset_revision = "61ec212baa25007bb06304c7dac8df0e96b1967b"
tasks = ("reconstruct", "generate", "edit")
run_dir = Path("runs/fine-tuning") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir.mkdir(parents=True)
hf_model_id = "lzhoang2601/Qwen3.5-4B-mathematical-tikz"
max_seq_length = 4096
max_length = 2048
max_new_tokens = 2048

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
import subprocess
import sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run([
    "apt-get", "install", "-y", "-qq", "--no-install-recommends",
    "texlive-xetex", "texlive-latex-extra", "texlive-pictures", "texlive-science", "lmodern", "latexmk",
], check=True)
%pip install -q httpx "pydantic>=2" "shapely>=2" pymupdf

repo_dir = Path.cwd()
if repo_dir.name == "train":
    repo_dir = repo_dir.parent
if not (repo_dir / "train/evaluate.py").is_file():
    repo_dir = Path("/tmp/mathematical_tikz")
    if not repo_dir.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/lzhoang2801/mathematical_tikz.git", str(repo_dir),
        ], check=True)
sys.path[:0] = [str(repo_dir), str(repo_dir / "src")]
from train.evaluate import evaluate_pair, local_compiler, observation

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

## Data and preview


In [ ]:
from datasets import load_dataset

dataset = {
    task: load_dataset(dataset_name, name=task, revision=dataset_revision)
    for task in tasks
}
print({task: {split: len(rows) for split, rows in splits.items()} for task, splits in dataset.items()})
(run_dir / "dataset.json").write_text(json.dumps({
    "dataset": dataset_name, "revision": dataset_revision,
}), encoding="utf-8")

In [ ]:
from IPython.display import Code, Image, Markdown, display

with local_compiler() as compiler:
    for task in tasks:
        if not len(dataset[task]["train"]):
            continue
        sample = dataset[task]["train"][0]
        display(Markdown(f"### {task}"))
        if task == "reconstruct":
            display(sample["image"])
        else:
            display(Markdown(sample["instruction"]))
        for name in (("current_tikz", "target_tikz") if task == "edit" else ("target_tikz",)):
            observed = observation(sample[name], run_dir / "preview" / task / name, compiler, preview=True)
            display(Image(filename=observed["image_path"]) if observed["image_path"] else observed["compile"])
            display(Code(sample[name], language="latex"))

## Task instructions and messages


In [ ]:
SYSTEM_PROMPTS = {
    "reconstruct": (
        "Reconstruct the mathematical diagram in the supplied image as TikZ code. "
        "Reproduce the visible geometry, relative layout, labels, mathematical notation, "
        "line styles and relation marks as faithfully as possible in black and white. "
        "Include only what is visible in the image; do not invent hidden constructions "
        "or additional mathematical facts. "
        "Return only TikZ source containing one complete tikzpicture. "
        "Do not include Markdown fences, explanations, or a LaTeX document wrapper."
        " Necessary TikZ library declarations may precede the picture."
    ),
    "generate": (
        "Create a mathematically accurate TikZ diagram from the user's givens and "
        "requested constructions. Respect named objects, incidences, lengths, ratios, "
        "angles, and specified intersection branches. Construct dependent points from "
        "their defining relations, include the requested auxiliary geometry, and keep "
        "labels and relation marks readable. Do not assume a conclusion to be proved. "
        "Return only TikZ source containing one complete tikzpicture. "
        "Do not include Markdown fences, explanations, or a LaTeX document wrapper."
        " Begin with the tikzpicture environment; the surrounding LaTeX wrapper supplies the libraries."
    ),
    "edit": (
        "Update the supplied current TikZ diagram according to the user's edit instruction. "
        "Preserve all unaffected geometry, labels, annotations, styles, and the existing "
        "coordinate frame. Construct the requested additions from the existing objects "
        "and update any affected dependencies while keeping the diagram mathematically "
        "consistent and readable. Return the complete updated diagram, including retained "
        "objects, rather than a patch or only the added code. "
        "Return only TikZ source containing one complete tikzpicture. "
        "Do not include Markdown fences, explanations, or a LaTeX document wrapper."
        " Begin with the tikzpicture environment; the surrounding LaTeX wrapper supplies the libraries."
    ),
}

def convert_to_conversation(sample, task, *, include_target=True):
    if task not in SYSTEM_PROMPTS:
        raise ValueError(f"Unknown task: {task}")
    if task == "reconstruct":
        if sample.get("image") is None:
            raise ValueError("Reconstruct requires an image.")
        content = [
            {"type": "image", "image": sample["image"]},
        ]
    else:
        instruction = sample.get("instruction")
        if not isinstance(instruction, str) or not instruction.strip():
            raise ValueError(f"{task} requires an instruction.")
        content = [{"type": "text", "text": instruction}]
        if task == "edit":
            current = sample.get("current_tikz")
            if not isinstance(current, str) or not current.strip():
                raise ValueError("Edit requires current_tikz.")
            content.append({"type": "text", "text": f"Current TikZ:\n```latex\n{current}\n```"})
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPTS[task]}]},
        {"role": "user", "content": content},
    ]
    if include_target:
        target = sample.get("target_tikz")
        if not isinstance(target, str) or not target.strip():
            raise ValueError("Training requires target_tikz.")
        messages.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return {"messages": messages}

In [ ]:
converted_dataset = [
    convert_to_conversation(sample, task)
    for task in tasks for sample in dataset[task]["train"]
]
converted_validation_dataset = [
    convert_to_conversation(sample, task)
    for task in tasks for sample in dataset[task]["validation"]
]
for task in tasks:
    print(task, SYSTEM_PROMPTS[task])
display(converted_dataset[0])

## Fine-tune


In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)
data_collator = UnslothVisionDataCollator(model, tokenizer)

for sample in converted_dataset + converted_validation_dataset:
    batch = data_collator([sample])
    text = tokenizer.batch_decode(batch["input_ids"], skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0]
    target = sample["messages"][-1]["content"][0]["text"]
    if batch["input_ids"].shape[-1] > max_length or not text.rstrip().endswith(target.rstrip()):
        raise ValueError("Complete conversation exceeds max_length, adjust it before training.")
del batch

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator,
    train_dataset = converted_dataset,
    eval_dataset = converted_validation_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = str(run_dir / "checkpoints"),
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = max_length,
        save_steps = 100,
        save_total_limit = 2,
    ),
)

In [ ]:
trainer_stats = trainer.train()
trainer.save_metrics("train", trainer_stats.metrics)
trainer.save_state()
trainer_stats.metrics

In [ ]:
model.save_pretrained(str(run_dir / "qwen_lora"))
tokenizer.save_pretrained(str(run_dir / "qwen_lora"))

## Inference


In [ ]:
FastVisionModel.for_inference(model)


def generate_tikz(sample, task):
    messages = convert_to_conversation(sample, task, include_target=False)["messages"]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False)
    images = [part["image"] for message in messages for part in message["content"] if part["type"] == "image"]
    inputs = tokenizer(text=[input_text], images=images or None,
                       add_special_tokens=False, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[-1]
    if input_length + max_new_tokens > max_seq_length:
        raise ValueError("Prompt plus generation budget exceeds max_seq_length.")
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tokenizer.batch_decode(output[:, input_length:], skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0]


def export_predictions(destination, split="validation", limit=5):
    if split not in ("validation", "test") or limit < 1:
        raise ValueError("Use validation/test with a positive per-task limit.")
    with Path(destination).open("x", encoding="utf-8") as stream:
        for task in tasks:
            rows = dataset[task][split]
            for row_index in range(min(limit, len(rows))):
                sample = rows[row_index]
                output_tikz = generate_tikz(sample, task)
                stream.write(json.dumps({
                    "task": task, "split": split, "row_index": row_index, "id": sample["id"],
                    "dataset_revision": dataset_revision,
                    "target_tikz": sample["target_tikz"], "output_tikz": output_tikz,
                }, ensure_ascii=False) + "\n")
                stream.flush()

In [ ]:
evaluation_split = "validation"
evaluation_limit = 5
predictions_path = run_dir / f"predictions-{evaluation_split}.jsonl"
export_predictions(predictions_path, split=evaluation_split, limit=evaluation_limit)

In [ ]:
from IPython.display import Code, Image, Markdown, display
from hashlib import sha256

predictions = [json.loads(line) for line in predictions_path.read_text(encoding="utf-8").splitlines()]
with local_compiler() as compiler:
    for task in tasks:
        prediction = next((row for row in predictions if row["task"] == task), None)
        if prediction is None:
            continue
        sample = dataset[task][evaluation_split][prediction["row_index"]]
        display(Markdown(f"### Fine-tuned {task}"))
        display(sample["image"] if task == "reconstruct" else Markdown(sample["instruction"]))
        output_tikz = prediction["output_tikz"]
        display(Code(output_tikz, language="latex"))
        observed = observation(
            output_tikz, run_dir / "observations" / sha256(output_tikz.encode()).hexdigest(),
            compiler, preview=True,
        )
        display(Image(filename=observed["image_path"]) if observed["image_path"] else observed["compile"])

## Evaluate


In [ ]:
from evaluation.batch import evaluate_file

scores_path = run_dir / f"scores-{evaluation_split}.jsonl"
with local_compiler() as compiler:
    summary = evaluate_file(
        predictions_path, scores_path,
        evaluator=lambda target, output: evaluate_pair(target, output, run_dir / "observations", compiler),
    )
display(summary)


## Save and upload to Hugging Face


In [ ]:
if False:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    model.push_to_hub_merged("lzhoang2601/qwen-tikz", tokenizer, token = hf_token)
    model.push_to_hub_gguf(
        "lzhoang2601/qwen-tikz",
        tokenizer,
        quantization_method = ["f16", "q4_k_m",],
        token = hf_token,
    )

## Reload


In [ ]:
if False:
    from unsloth import FastVisionModel
    import torch

    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = hf_model_id,
        max_seq_length = max_seq_length,
        load_in_4bit = False,
    )
    FastVisionModel.for_inference(model)